# Logit Lens: Tracing Predictions Through Depth

**Survey Reference:** Section 9.1 - Logit Lens: Tracing Predictions Through Depth

## Overview

The **Logit Lens** is a simple but powerful technique that projects intermediate representations to vocabulary space using the final unembedding matrix. This reveals how predictions "form" as information flows through layers.

### The Core Idea

At each layer, apply the unembedding:

$$\text{LogitLens}(h_l) = h_l \cdot W_U$$

where $h_l$ is the residual stream at layer $l$ and $W_U$ is the unembedding matrix.

## Learning Objectives

1. Implement the logit lens from scratch
2. Visualize how predictions evolve across layers
3. Identify when key information enters the residual stream
4. Understand limitations of the logit lens

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Loading a Model

In [ ]:
# Load a small model for demonstration
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Layers: {model.config.n_layer}")
print(f"Hidden size: {model.config.n_embd}")
print(f"Vocab size: {model.config.vocab_size}")

## 2. Extracting Hidden States

In [ ]:
def get_all_hidden_states(model, tokenizer, text: str):
    """
    Get hidden states from all layers.
    
    Returns:
        hidden_states: List of [1, seq_len, hidden_size] tensors
        tokens: List of token strings
    """
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    hidden_states = outputs.hidden_states  # Tuple of (n_layers + 1) tensors
    tokens = [tokenizer.decode(t) for t in inputs.input_ids[0]]
    
    return hidden_states, tokens

## 3. Implementing the Logit Lens

In [ ]:
def logit_lens(hidden_states, model):
    """
    Apply logit lens to all hidden states.
    
    Projects each layer's hidden state to vocabulary space.
    
    Returns:
        logits: [n_layers, seq_len, vocab_size]
    """
    # Get the unembedding matrix (and layer norm if applicable)
    ln_f = model.transformer.ln_f  # Final layer norm
    unembed = model.lm_head.weight  # [vocab_size, hidden_size]
    
    all_logits = []
    
    for layer_idx, h in enumerate(hidden_states):
        # Apply final layer norm (important for GPT-2 style models)
        h_normed = ln_f(h)
        
        # Project to vocabulary
        logits = h_normed @ unembed.T  # [1, seq_len, vocab_size]
        all_logits.append(logits[0])  # Remove batch dim
    
    return torch.stack(all_logits)  # [n_layers, seq_len, vocab_size]

In [ ]:
# Test on a simple example
text = "The capital of France is"
hidden_states, tokens = get_all_hidden_states(model, tokenizer, text)

print(f"Tokens: {tokens}")
print(f"Number of layers (including embedding): {len(hidden_states)}")

In [ ]:
# Apply logit lens
with torch.no_grad():
    all_logits = logit_lens(hidden_states, model)

print(f"Logits shape: {all_logits.shape}")

# Get top predictions at last position for each layer
last_pos_logits = all_logits[:, -1, :]  # [n_layers, vocab_size]
top_tokens = last_pos_logits.argmax(dim=-1)

print("\nTop prediction at each layer:")
for layer_idx, tok_id in enumerate(top_tokens):
    token_str = tokenizer.decode(tok_id)
    prob = F.softmax(last_pos_logits[layer_idx], dim=-1)[tok_id].item()
    print(f"  Layer {layer_idx:2d}: '{token_str}' (prob: {prob:.4f})")

## 4. Visualizing Prediction Evolution

In [ ]:
def visualize_logit_lens(all_logits, tokens, tokenizer, target_tokens=None, top_k=5):
    """
    Visualize how predictions evolve across layers.
    
    Args:
        target_tokens: Specific tokens to track (if None, use top predictions)
    """
    n_layers = all_logits.shape[0]
    
    # Focus on last position (next token prediction)
    last_logits = all_logits[:, -1, :]  # [n_layers, vocab_size]
    probs = F.softmax(last_logits, dim=-1)
    
    # Get final top-k tokens to track
    if target_tokens is None:
        final_top_k = probs[-1].topk(top_k).indices
        target_tokens = [tokenizer.decode(t) for t in final_top_k]
        target_ids = final_top_k
    else:
        target_ids = [tokenizer.encode(t)[0] for t in target_tokens]
    
    # Plot probability evolution
    plt.figure(figsize=(10, 6))
    
    for tok_str, tok_id in zip(target_tokens, target_ids):
        tok_probs = probs[:, tok_id].cpu().numpy()
        plt.plot(range(n_layers), tok_probs, marker='o', label=f"'{tok_str}'")
    
    plt.xlabel('Layer')
    plt.ylabel('Probability')
    plt.title(f'Logit Lens: Prediction Evolution\nInput: "{" ".join(tokens)}"')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

visualize_logit_lens(all_logits, tokens, tokenizer)

## 5. Layer-by-Layer Analysis

In [ ]:
def logit_lens_heatmap(all_logits, tokens, tokenizer, top_k=10):
    """
    Create a heatmap showing top-k token probabilities across layers.
    """
    n_layers = all_logits.shape[0]
    last_logits = all_logits[:, -1, :]
    probs = F.softmax(last_logits, dim=-1)
    
    # Get union of top-k tokens across all layers
    all_top_tokens = set()
    for layer in range(n_layers):
        top_indices = probs[layer].topk(top_k).indices.cpu().tolist()
        all_top_tokens.update(top_indices)
    
    all_top_tokens = sorted(list(all_top_tokens))
    
    # Build probability matrix
    prob_matrix = np.zeros((n_layers, len(all_top_tokens)))
    for i, tok_id in enumerate(all_top_tokens):
        prob_matrix[:, i] = probs[:, tok_id].cpu().numpy()
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(14, 8))
    im = ax.imshow(prob_matrix.T, aspect='auto', cmap='viridis')
    
    # Labels
    ax.set_xlabel('Layer')
    ax.set_ylabel('Token')
    ax.set_yticks(range(len(all_top_tokens)))
    ax.set_yticklabels([tokenizer.decode(t) for t in all_top_tokens])
    
    plt.colorbar(im, label='Probability')
    plt.title('Logit Lens Heatmap')
    plt.tight_layout()
    plt.show()

logit_lens_heatmap(all_logits, tokens, tokenizer)

## 6. Limitations of the Logit Lens

The logit lens has known limitations:

1. **Representation Drift:** Early layers may use a different "basis" than the unembedding expects
2. **Layer Norm:** Applying final layer norm may not be appropriate for early layers
3. **Confidence Calibration:** Probabilities in early layers are poorly calibrated

→ These limitations motivate the **Tuned Lens** (next notebook)

In [ ]:
# Demonstrate representation drift
def measure_representation_drift(hidden_states):
    """
    Measure how much the representation changes between layers.
    """
    n_layers = len(hidden_states)
    drifts = []
    
    for i in range(1, n_layers):
        # Cosine similarity between consecutive layers
        h_prev = hidden_states[i-1][:, -1, :]  # Last position
        h_curr = hidden_states[i][:, -1, :]
        
        cos_sim = F.cosine_similarity(h_prev, h_curr, dim=-1).item()
        drifts.append(1 - cos_sim)  # Drift = 1 - similarity
    
    return drifts

drifts = measure_representation_drift(hidden_states)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(drifts)+1), drifts, marker='o')
plt.xlabel('Layer')
plt.ylabel('Representation Drift (1 - cosine sim)')
plt.title('Representation Drift Across Layers')
plt.grid(True, alpha=0.3)
plt.show()

## Key Takeaways

1. **Logit lens** projects intermediate states to vocabulary using the unembedding
2. **Predictions form gradually** as information flows through layers
3. **Key transitions** often happen in middle layers
4. **Limitations exist:** Representation drift makes early layers harder to interpret
5. **Layer norm matters:** Different choices give different results

## Next Steps

→ **02_tuned_lens.ipynb**: Learn per-layer transforms for better interpretation